# TP 1 — Apprentissage supervisé 2A (correction)

**Contenu du sujet :**
- **Partie 1** : règle de Bayes et risque de Bayes en classification binaire (TD).
- **Partie 2** : simulation de données, jeux réels, algorithme des *k* plus proches voisins (k-NN).

> Les sujets sont rédigés pour R (packages `mlbench`, etc.). Cette correction utilise **Python / scikit-learn**,
> qui offre les équivalents directs. Les jeux de données réels sont téléchargés automatiquement depuis **OpenML**.


## Brief de démarrage — Les outils de ce TP

> **À lire avant de commencer.** Ce premier TP pose les **fondations** du cours : la référence théorique (la
> règle de Bayes) et le premier vrai classifieur (le k-NN). Voici comment ils fonctionnent.

**1. Règle de Bayes et risque de Bayes — la référence *optimale*.**
En classification binaire avec le coût 0–1, la **meilleure décision possible** consiste à prédire la classe la
plus probable :
$$g^\star(x) = \mathbb{1}\{\eta(x) \ge 1/2\}, \qquad \eta(x) = \mathbb{P}(Y = 1 \mid X = x).$$
Son taux d'erreur, le **risque de Bayes** $L^\star = \mathbb{E}[\min(\eta(X), 1-\eta(X))]$, est le **plancher**
que **aucun** modèle ne peut battre. On ne l'atteint jamais en pratique (on ne connaît pas $\eta$), mais il sert
de **point de comparaison** : un bon modèle s'en approche.

**2. k plus proches voisins (k-NN) — estimer $\eta$ *localement*.**
Idée : $\eta(x)$ est estimé par la **proportion de voisins** de la classe 1 parmi les $k$ points les plus
proches de $x$. On classe ensuite au vote majoritaire.
- **$k$ petit** (ex. 1) : colle aux données, frontière très irrégulière ⇒ **variance élevée** (surajustement) ;
- **$k$ grand** : moyenne sur un large voisinage, frontière lisse ⇒ **biais élevé** (sous-ajustement).
- Le k-NN mesure des **distances** : les variables doivent être à la **même échelle** (standardisation).

Le bon $k$ réalise le meilleur **compromis biais–variance** ; on le choisit en comparant l'erreur sur un jeu de
test (ou par validation croisée).

> **Méthode du TP.** (Partie 1) calculer la règle et le risque de Bayes sur un modèle connu ; (Partie 2)
> simuler des données puis appliquer le k-NN, faire **varier $k$** et observer son effet sur l'erreur.

In [1]:
import sys

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.datasets import fetch_openml
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sys.path += [".", ".."]          # plot_theme.py est a la racine du projet
from plot_theme import *         # couleurs, pick_color(), theme sombre applique par defaut

SEED = 42
np.random.seed(SEED)             # graine unique posee une fois : tout le TP est reproductible

---

# 🟦 PARTIE 1 — Règle de Bayes et risque en classification binaire

---

### Rappel de cours

On observe un couple $(X, Y)$ où $Y \in \{0, 1\}$ (classification **binaire**). Un **classifieur** est une
fonction $g : \mathcal{X} \to \{0,1\}$. On mesure sa qualité par le **risque** (probabilité d'erreur) associé au
coût 0–1 :
$$L(g) = \mathbb{P}\big(g(X) \neq Y\big).$$

L'objet central est la **fonction de régression** (probabilité a posteriori) :
$$\eta(x) = \mathbb{P}(Y = 1 \mid X = x).$$

Le classifieur qui **minimise** le risque est le **classifieur de Bayes** :
$$g^\star(x) = \mathbb{1}\big\{\eta(x) \ge \tfrac{1}{2}\big\}
= \begin{cases} 1 & \text{si } \eta(x) \ge 1/2 \\ 0 & \text{sinon.} \end{cases}$$

Autrement dit : *on prédit la classe la plus probable au point $x$*. Son risque, le **risque de Bayes**, est le
plus petit risque atteignable :
$$L^\star = L(g^\star) = \mathbb{E}\big[\min\big(\eta(X),\, 1 - \eta(X)\big)\big].$$

**Intuition.** En un point $x$, la meilleure décision se trompe avec probabilité $\min(\eta(x), 1-\eta(x))$
(on parie sur la classe majoritaire, on perd si c'est l'autre). On moyenne ensuite sur la loi de $X$.



**Exemple express.** Si $\eta(x) = 0.9$, on prédit $1$ et on se trompe dans $10\%$ des cas *à ce point*. Si $\eta(x) = 0.5$, aucune information : on se trompe 1 fois sur 2 (situation la plus difficile).

### Cas 1

Modèle : $X_i \sim \mathcal{N}(0,1)$, $U_i \sim \mathcal{U}[0,1]$ indépendants, et
$$Y_i = \begin{cases} \mathbb{1}(U_i \le 0.1) & \text{si } X_i \le 0 \\ \mathbb{1}(U_i > 0.2) & \text{si } X_i > 0. \end{cases}$$

**Étape 1 — Calcul de $\eta$.** Comme $U$ est indépendant de $X$ :
- si $x \le 0$ : $\eta(x) = \mathbb{P}(U \le 0.1) = 0.1$ ;
- si $x > 0$ : $\eta(x) = \mathbb{P}(U > 0.2) = 0.8$.

**Étape 2 — Règle de Bayes** ($\eta \ge 1/2$ ?) :
- $x \le 0$ : $\eta = 0.1 < 0.5 \Rightarrow g^\star(x) = 0$ ;
- $x > 0$ : $\eta = 0.8 \ge 0.5 \Rightarrow g^\star(x) = 1$.

Donc $g^\star(x) = \mathbb{1}\{x > 0\}$.

**Étape 3 — Risque de Bayes.** Comme $X \sim \mathcal{N}(0,1)$, $\mathbb{P}(X \le 0) = \mathbb{P}(X > 0) = 0.5$ :
$$L^\star = 0.5 \cdot \min(0.1, 0.9) + 0.5 \cdot \min(0.8, 0.2) = 0.5(0.1) + 0.5(0.2) = \boxed{0.15}.$$


### Cas 2

Même modèle avec les seuils $0.2$ et $0.4$ :
$$Y_i = \begin{cases} \mathbb{1}(U_i \le 0.2) & \text{si } X_i \le 0 \\ \mathbb{1}(U_i > 0.4) & \text{si } X_i > 0. \end{cases}$$

- $x \le 0$ : $\eta(x) = 0.2$ ; $x > 0$ : $\eta(x) = \mathbb{P}(U > 0.4) = 0.6$.
- Règle de Bayes : $x \le 0 \Rightarrow 0$ (car $0.2 < 0.5$) ; $x > 0 \Rightarrow 1$ (car $0.6 \ge 0.5$).
  Donc $g^\star(x) = \mathbb{1}\{x > 0\}$ (même règle que le Cas 1).
- Risque de Bayes :
$$L^\star = 0.5 \cdot \min(0.2, 0.8) + 0.5 \cdot \min(0.6, 0.4) = 0.5(0.2) + 0.5(0.4) = \boxed{0.30}.$$


### Cas 3 (données réelles « café »)

Ici $X$ = sexe (femme / homme) et $Y$ = « café sucré » (1) ou non (0). L'énoncé donne :
$$\mathbb{P}(Y=1 \mid \text{femme}) = 0.2, \qquad \mathbb{P}(Y=1 \mid \text{homme}) = 0.1.$$

**Règle de Bayes.** Les deux probabilités sont $< 0.5$ : quel que soit le sexe, la classe la plus probable est
« non sucré ». Donc
$$g^\star(x) = 0 \quad \text{pour tout } x \;\; (\text{on prédit toujours « non sucré »}).$$

**Risque de Bayes.** Il vaut
$$L^\star = \mathbb{P}(Y = 1) = \mathbb{P}(\text{femme})\,(0.2) + \mathbb{P}(\text{homme})\,(0.1).$$
⚠️ **On ne peut pas donner de valeur numérique** : l'énoncé ne fournit **pas** la répartition
femmes / hommes $\mathbb{P}(\text{femme})$. La règle de Bayes est parfaitement définie, **mais le risque de
Bayes reste incalculable** sans cette information. Si l'échantillon était équilibré ($50\%/50\%$), on aurait
$L^\star = 0.5(0.2) + 0.5(0.1) = 0.15$.


### Vérification numérique par simulation (Monte-Carlo)

On simule un grand échantillon et on estime empiriquement le risque de Bayes en appliquant la règle
$g^\star(x) = \mathbb{1}\{x > 0\}$. On doit retrouver $\approx 0.15$ (Cas 1) et $\approx 0.30$ (Cas 2).


In [2]:
def simule_cas(n, a, b):
    """Modele du sujet : Y = 1(U <= a) si X <= 0, sinon Y = 1(U > b)."""
    X = np.random.normal(size=n)
    U = np.random.uniform(size=n)
    Y = np.where(X <= 0, U <= a, U > b).astype(int)
    return X, Y


def risque_bayes_empirique(n, a, b):
    X, Y = simule_cas(n, a, b)
    return np.mean((X > 0).astype(int) != Y)   # regle de Bayes g*(x) = 1{x > 0}


N = 500_000
for cas, a, b, theorique in [("Cas 1", 0.1, 0.2, 0.15), ("Cas 2", 0.2, 0.4, 0.30)]:
    print(f"{cas} : risque empirique = {risque_bayes_empirique(N, a, b):.4f}  (theorique {theorique:.2f})")

Cas 1 : risque empirique = 0.1498  (theorique 0.15)
Cas 2 : risque empirique = 0.3010  (theorique 0.30)


### Question 2 — Indicateur de complexité et classement

Le **risque de Bayes $L^\star$** est l'indicateur naturel de difficulté : c'est l'erreur **incompressible**,
que même le meilleur classifieur possible ne peut pas descendre en dessous. Plus $L^\star$ est grand, plus le
problème est difficile (le bruit domine). De manière équivalente, un problème est **facile** quand $\eta(x)$ est
**loin de $1/2$** (classes bien séparées), et **difficile** quand $\eta(x)$ est **proche de $1/2$**.

**Classement (du plus facile au plus difficile) :**
$$\underbrace{\text{Cas 1}}_{L^\star = 0.15} \;<\; \underbrace{\text{Cas 3}}_{L^\star = 0.15 \text{ si équilibré}} \;<\; \underbrace{\text{Cas 2}}_{L^\star = 0.30}.$$
Le Cas 2 est le plus difficile. Le Cas 3 ne peut être classé précisément faute de connaître la répartition
hommes/femmes, mais sous hypothèse d'équilibre il est de difficulté comparable au Cas 1.


---

# 🟩 PARTIE 2 — Manipulation de données et algorithme des k plus proches voisins

---

### Rappel de cours — Règle de décision du k-NN

Pour classer un nouveau point $x$, l'algorithme des **$k$ plus proches voisins** :
1. calcule la distance (souvent **euclidienne**) de $x$ à tous les points d'apprentissage ;
2. retient les **$k$** points les plus proches ;
3. prédit la classe **majoritaire** parmi ces $k$ voisins (**vote**).

Formellement, $\hat{g}(x) = \arg\max_{c} \sum_{i \in V_k(x)} \mathbb{1}\{y_i = c\}$, où $V_k(x)$ est l'ensemble
des indices des $k$ plus proches voisins. C'est une méthode **non paramétrique** : elle ne suppose aucune forme
pour $\eta$, mais stocke tout l'échantillon.

**Deux points de vigilance** (repris en Exercice 2) :
- le choix de **$k$** (hyperparamètre) contrôle le compromis biais–variance ($k$ petit ⇒ surajustement ;
  $k$ grand ⇒ sous-ajustement) ;
- la méthode est **sensible à l'échelle** des variables (les distances sont dominées par les variables de
  grande amplitude) ⇒ il faut **standardiser**.

### Exercice 1 — Situation du Cas 2

**1.1 & 1.2** On simule un échantillon d'**apprentissage** de taille $1000$ et un échantillon **indépendant** de
taille $200$. Ce second échantillon sert de **jeu de test** : il permet d'estimer l'erreur de **généralisation**
(sur des données jamais vues) sans biais optimiste, contrairement à l'erreur mesurée sur l'apprentissage.


In [3]:
# 1.1 apprentissage (n = 1000) et 1.2 test (n = 200), Cas 2 : a = 0.2, b = 0.4
X_train, y_train = simule_cas(1000, 0.2, 0.4)
X_test, y_test = simule_cas(200, 0.2, 0.4)

# scikit-learn attend une matrice 2D (n_observations, n_variables) : ici 1 variable
X_train, X_test = X_train.reshape(-1, 1), X_test.reshape(-1, 1)
print("Apprentissage :", X_train.shape, "| Test :", X_test.shape)

Apprentissage : (1000, 1) | Test : (200, 1)


**2 & 3** On entraîne un k-NN **sans réglage** (valeur par défaut $k = 5$) et on évalue l'erreur sur le test.
Le risque de Bayes du Cas 2 étant $0.30$, on s'attend à une erreur de test proche de cette borne : le k-NN ne
peut pas faire mieux que $\approx 0.30$ en moyenne.


In [4]:
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)   # k = 5 par defaut, aucun reglage

err_train = 1 - knn.score(X_train, y_train)
err_test = 1 - knn.score(X_test, y_test)
print(f"Erreur d'apprentissage : {err_train:.3f}")
print(f"Erreur de test         : {err_test:.3f}   (risque de Bayes = 0.30)")

Erreur d'apprentissage : 0.230
Erreur de test         : 0.315   (risque de Bayes = 0.30)


**Lecture.** L'erreur de test est du **même ordre que le risque de Bayes** $L^\star = 0.30$, en général un peu
au-dessus, et ne peut pas descendre durablement en dessous. Trois remarques :
- avec $k = 5$, l'estimation locale de $\eta$ repose sur 5 votes seulement : elle est **bruitée**, d'où un
  **excès de risque** par rapport à la règle de Bayes ;
- avec seulement $200$ points de test, l'erreur estimée **fluctue** : son écart-type vaut environ
  $\sqrt{0.3 \times 0.7 / 200} \approx 0.03$. Un écart de quelques points à $0.30$ n'est donc pas significatif ;
- l'erreur d'apprentissage est plus **faible** (le modèle « voit » ces points, et chaque point est son propre
  voisin) : il ne faut **jamais** évaluer un modèle sur ses données d'entraînement.

**Effet de $k$.** Le graphique suivant fait varier $k$ : pour $k = 1$, l'erreur d'apprentissage est nulle mais
l'erreur de test est forte (théorème de Cover–Hart : asymptotiquement, $L_{1\text{-NN}} \le 2L^\star(1-L^\star)
= 0.42$) ; quand $k$ augmente, l'estimation locale de $\eta$ se stabilise et l'erreur de test se rapproche de
$L^\star = 0.30$. Si $k$ devient **trop grand**, le voisinage déborde de part et d'autre de $x = 0$ et l'erreur
remonte près de la frontière (biais).

In [5]:
ks = np.arange(1, 202, 4)
erreurs = {"apprentissage": [], "test": []}
for k in ks:
    modele = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    erreurs["apprentissage"].append(1 - modele.score(X_train, y_train))
    erreurs["test"].append(1 - modele.score(X_test, y_test))

fig = go.Figure()
for i, (nom, err) in enumerate(erreurs.items()):
    fig.add_trace(go.Scatter(x=ks, y=err, mode="lines+markers", name=f"Erreur {nom}",
                             line=dict(color=pick_color(i))))
fig.add_hline(y=0.30, line_dash="dash", line_color=HIGHLIGHT, annotation_text="Risque de Bayes = 0.30")
fig.update_layout(title="Cas 2 : erreur du k-NN en fonction de k", xaxis_title="k (nombre de voisins)",
                  yaxis_title="Taux d'erreur")
fig.show()

### Exercice 2 — Jeu de données réel « Vehicle »

Le sujet R utilise `mlbench::Vehicle`. L'équivalent Python est le jeu **Vehicle (Statlog)** disponible sur
OpenML : il s'agit de reconnaître le **type de véhicule** (4 classes) à partir de variables de forme extraites
de silhouettes.

**1.1 & 1.2** Chargement et description.


In [6]:
vehicle = fetch_openml("vehicle", version=1, as_frame=True)
X = vehicle.data
y = vehicle.target

print("Dimensions :", X.shape)
print("Classes    :", sorted(y.unique()))
print(X.head())


Dimensions : (846, 18)
Classes    : ['bus', 'opel', 'saab', 'van']
   COMPACTNESS  CIRCULARITY  DISTANCE_CIRCULARITY  RADIUS_RATIO  \
0           95           48                    83           178   
1           91           41                    84           141   
2          104           50                   106           209   
3           93           41                    82           159   
4           85           44                    70           205   

   PR.AXIS_ASPECT_RATIO  MAX.LENGTH_ASPECT_RATIO  SCATTER_RATIO  \
0                    72                       10            162   
1                    57                        9            149   
2                    66                       10            207   
3                    63                        9            144   
4                   103                       52            149   

   ELONGATEDNESS  PR.AXIS_RECTANGULARITY  MAX.LENGTH_RECTANGULARITY  \
0             42                      20                

In [7]:
# Distribution des classes
effectifs = y.value_counts().sort_index()
fig = go.Figure(go.Bar(x=effectifs.index, y=effectifs.values, text=effectifs.values,
                       textposition="outside", marker_color=pick_color(0)))
fig.update_layout(title="Distribution des classes du jeu Vehicle", xaxis_title="Classe", yaxis_title="Effectif")
fig.show()

In [8]:
# 1.2 description du jeu de donnees
print(vehicle.DESCR)

**Author**: Dr. Pete Mowforth and Dr. Barry Shepherd  
**Source**: [UCI](https://archive.ics.uci.edu/ml/datasets/Statlog+(Vehicle+Silhouettes))
**Please cite**: Siebert,JP. Turing Institute Research Memorandum TIRM-87-018 "Vehicle Recognition Using Rule Based Methods" (March 1987)  

 NAME
         vehicle silhouettes
 
 PURPOSE
         to classify a given silhouette as one of four types of vehicle,
         using  a set of features extracted from the silhouette. The
         vehicle may be viewed from one of many different angles.  
 
 PROBLEM TYPE
         classification
         
 SOURCE
         Drs.Pete Mowforth and Barry Shepherd
         Turing Institute
         George House
         36 North Hanover St.
         Glasgow
         G1 2AD
 
 CONTACT
         Alistair Sutherland
         Statistics Dept.
         Strathclyde University
         Livingstone Tower
         26 Richmond St.
         GLASGOW G1 1XH
         Great Britain
         
         Tel: 041 552 4400 x3033
    

**1.3 Combien de prédicteurs potentiels ?** Le jeu contient **18** variables de forme (toutes les colonnes de
`X`), la cible `Class` étant à part.

**1.4 Sélection** : on retire les variables dont le nom commence par « Skewness » ou « Kurtosis »
(asymétrie / aplatissement), en ignorant la casse. Il en reste **14**.


In [9]:
print("Nombre de predicteurs potentiels :", X.shape[1])

a_exclure = [c for c in X.columns if c.upper().startswith(("SKEWNESS", "KURTOSIS"))]
X_sel = X.drop(columns=a_exclure)

print("Variables exclues :", a_exclure)
print("Predicteurs conserves :", X_sel.shape[1], "->", list(X_sel.columns))


Nombre de predicteurs potentiels : 18
Variables exclues : ['SKEWNESS_ABOUT_MAJOR', 'SKEWNESS_ABOUT_MINOR', 'KURTOSIS_ABOUT_MAJOR', 'KURTOSIS_ABOUT_MINOR']
Predicteurs conserves : 14 -> ['COMPACTNESS', 'CIRCULARITY', 'DISTANCE_CIRCULARITY', 'RADIUS_RATIO', 'PR.AXIS_ASPECT_RATIO', 'MAX.LENGTH_ASPECT_RATIO', 'SCATTER_RATIO', 'ELONGATEDNESS', 'PR.AXIS_RECTANGULARITY', 'MAX.LENGTH_RECTANGULARITY', 'SCALED_VARIANCE_MAJOR', 'SCALED_VARIANCE_MINOR', 'SCALED_RADIUS_OF_GYRATION', 'HOLLOWS_RATIO']


In [10]:
# Distribution des variables conservees
n_cols = 3
n_rows = int(np.ceil(X_sel.shape[1] / n_cols))
fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=list(X_sel.columns),
                    vertical_spacing=0.08, horizontal_spacing=0.08)
for i, var in enumerate(X_sel.columns):
    fig.add_trace(go.Histogram(x=X_sel[var], nbinsx=20, marker_color=pick_color(i), showlegend=False),
                  row=i // n_cols + 1, col=i % n_cols + 1)
fig.update_layout(title="Distribution des variables descriptives conservees (Vehicle)", height=280 * n_rows)
fig.show()

**2. Fonction de coût.** Problème de classification (multi-classes, 4 catégories) ⇒ on utilise le **coût 0–1**,
c'est-à-dire le **taux d'erreur de classification** $\frac{1}{n}\sum_i \mathbb{1}\{\hat{y}_i \neq y_i\}$
(de façon équivalente, on maximise l'*accuracy*).

**3.1 Première étape obligatoire.** Toujours **séparer les données en apprentissage / test** (ici avec
`stratify` pour conserver les proportions de classes). Le test ne doit **jamais** servir à l'entraînement ni au
réglage.


In [11]:
X_train, X_test, y_train, y_test = train_test_split(X_sel, y, test_size=0.25, stratify=y)
print("Apprentissage :", X_train.shape, "| Test :", X_test.shape)

Apprentissage : (634, 14) | Test : (212, 14)


**3.2 Sensibilité du k-NN & transformation.** Le k-NN repose sur des **distances** : une variable de grande
amplitude écrase les autres. La description (question 1.2) montre des variables d'**échelles très différentes**.
⇒ On **standardise** (centrer-réduire) chaque variable. Pour éviter toute **fuite de données**, on encapsule le
`StandardScaler` et le k-NN dans un **`Pipeline`** : la standardisation est apprise **uniquement** sur
l'apprentissage à chaque étape de validation croisée.


**3.3 Hyperparamètre à régler : $k$.** On l'optimise par **validation croisée** (`GridSearchCV`, 5 folds) sur
une grille de valeurs impaires (nombres impairs pour éviter les égalités de vote). On retient le $k$ qui
maximise l'accuracy en validation croisée, puis on mesure l'erreur sur le **test**.

**Pourquoi on ne fixe pas $k=4$ alors qu'il y a 4 classes ?**
- Le **nombre de classes** et le **nombre de voisins** sont deux choses différentes.
- $k$ contrôle le compromis biais-variance : petit $k$ = modèle très local (variance élevée), grand $k$ =
  modèle plus lisse (biais plus élevé).
- En multi-classes, $k$ n'a **pas** besoin d'être égal au nombre de classes.
- On prend souvent un $k$ impair (5, 7, 9, ...) pour limiter les ex-aequo dans le vote.

Ici, `k=5` n'est donc **pas** lié au fait qu'il y a 4 classes : c'est un choix de départ raisonnable, ensuite
confirmé (ou non) par validation croisée.

In [12]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier()),
])

grille = {"knn__n_neighbors": list(range(1, 31, 2))}
recherche = GridSearchCV(pipe, grille, cv=5, scoring="accuracy")
recherche.fit(X_train, y_train)

k_opt = recherche.best_params_["knn__n_neighbors"]
print(f"Meilleur k (validation croisee) : {k_opt}")
print(f"Accuracy CV au meilleur k       : {recherche.best_score_:.3f}")


Meilleur k (validation croisee) : 7
Accuracy CV au meilleur k       : 0.741


In [13]:
# Accuracy de validation croisee en fonction de k
ks = grille["knn__n_neighbors"]
scores = recherche.cv_results_["mean_test_score"]

fig = go.Figure(go.Scatter(x=ks, y=scores, mode="lines+markers", name="Accuracy CV",
                           line=dict(width=3, color=pick_color(0)), marker=dict(size=8, color=pick_color(2))))
fig.add_vline(x=k_opt, line_dash="dash", line_color=pick_color(1),
              annotation_text=f"k optimal = {k_opt}")
fig.update_layout(title="Reglage de k par validation croisee - Vehicle",
                  xaxis_title="k (nombre de voisins)", yaxis_title="Accuracy (validation croisee)")
fig.show()

In [14]:
# Erreur de prediction finale sur le jeu de test
err_test_vehicle = 1 - recherche.score(X_test, y_test)
print(f"Erreur de prediction (test) avec k = {k_opt} : {err_test_vehicle:.3f}")


Erreur de prediction (test) avec k = 7 : 0.278


**Conclusion de l'Exercice 2.**
- La **standardisation** est indispensable pour le k-NN (sinon les distances sont biaisées par l'échelle).
- Le **$k$ optimal** est choisi par validation croisée : il réalise le meilleur compromis biais–variance sur
  des données de validation, indépendamment du test.
- L'erreur finale est mesurée **une seule fois** sur le jeu de test, donnant une estimation honnête de la
  performance de généralisation.


---

# 📎 ANNEXE A — Exemple visuel simple pour comprendre le k-NN

---

**1. Le vote des voisins.** On construit un petit nuage de 10 points sur 2 axes, on ajoute un nouveau point
(étoile) et on le classe par k-NN :
- on regarde ses $k$ voisins les plus proches ;
- on prend la classe majoritaire parmi ces voisins.

**2. Pourquoi standardiser ?** Un cas concret (clients décrits par leur âge et leur revenu) montre que le plus
proche voisin **change** selon que l'on standardise ou non les variables.

In [15]:
# 1) Exemple visuel : 10 points, un nouveau point, k = 3
X_demo = np.array([
    [1.0, 1.1], [1.4, 1.3], [1.1, 1.8], [0.7, 1.5], [1.8, 0.9],
    [4.3, 3.8], [3.9, 4.4], [4.6, 4.2], [3.7, 3.6], [4.2, 4.8],
])
y_demo = np.array([0] * 5 + [1] * 5)
x_new = np.array([[2.8, 2.9]])
k = 3

pred = int(KNeighborsClassifier(n_neighbors=k).fit(X_demo, y_demo).predict(x_new)[0])
voisins = np.argsort(np.linalg.norm(X_demo - x_new, axis=1))[:k]

fig = go.Figure()
for i in voisins:   # segments vers les k voisins
    fig.add_trace(go.Scatter(x=[x_new[0, 0], X_demo[i, 0]], y=[x_new[0, 1], X_demo[i, 1]], mode="lines",
                             line=dict(color=MUTED, dash="dash"), showlegend=False))
for c in (0, 1):
    fig.add_trace(go.Scatter(x=X_demo[y_demo == c, 0], y=X_demo[y_demo == c, 1], mode="markers",
                             name=f"Classe {c}", marker=dict(size=12, color=pick_class_color(c))))
fig.add_trace(go.Scatter(x=X_demo[voisins, 0], y=X_demo[voisins, 1], mode="markers",
                         name=f"{k} plus proches voisins",
                         marker=dict(size=22, color="rgba(0,0,0,0)", line=dict(width=2, color=HIGHLIGHT))))
fig.add_trace(go.Scatter(x=x_new[:, 0], y=x_new[:, 1], mode="markers", name="Nouveau point",
                         marker=dict(size=18, symbol="star", color=pick_class_color("nouveau"))))
fig.update_layout(title=f"k-NN simple : k={k}, prediction = classe {pred}", xaxis_title="Axe 1", yaxis_title="Axe 2")
fig.show()

In [16]:
# 2) Pourquoi standardiser ? Clients decrits par (age, revenu annuel)
X_clients = np.array([[25, 25000], [30, 30000], [65, 61000], [70, 66000]])
noms = np.array(["A (jeune)", "B (jeune)", "C (senior)", "D (senior)"])
x_client = np.array([[27, 60000]])   # client jeune, mais revenu eleve

scaler = StandardScaler().fit(X_clients)
espaces = {
    "Echelle brute : le revenu domine": (X_clients, x_client, ""),
    "Apres standardisation : age et revenu comptent": (scaler.transform(X_clients), scaler.transform(x_client),
                                                       " (standardise)"),
}

fig = make_subplots(rows=1, cols=2, subplot_titles=list(espaces))
for col, (titre, (Xc, xq, suffixe)) in enumerate(espaces.items(), start=1):
    i = int(np.argmin(np.linalg.norm(Xc - xq, axis=1)))
    print(f"{titre:48s} -> plus proche voisin : {noms[i]}")
    fig.add_trace(go.Scatter(x=[xq[0, 0], Xc[i, 0]], y=[xq[0, 1], Xc[i, 1]], mode="lines",
                             line=dict(color=HIGHLIGHT, dash="dash"), showlegend=False), row=1, col=col)
    fig.add_trace(go.Scatter(x=Xc[:, 0], y=Xc[:, 1], mode="markers+text", text=noms, textposition="top center",
                             marker=dict(size=14, color=[pick_class_color(0)] * 2 + [pick_class_color(1)] * 2),
                             showlegend=False), row=1, col=col)
    fig.add_trace(go.Scatter(x=xq[:, 0], y=xq[:, 1], mode="markers", name="Nouveau client", showlegend=col == 1,
                             marker=dict(size=20, symbol="star", color=pick_class_color("nouveau"))),
                  row=1, col=col)
    fig.update_xaxes(title_text="Age" + suffixe, row=1, col=col)
    fig.update_yaxes(title_text="Revenu annuel" + suffixe, row=1, col=col)
fig.update_layout(title="Le plus proche voisin change apres standardisation")
fig.show()

Echelle brute : le revenu domine                 -> plus proche voisin : C (senior)
Apres standardisation : age et revenu comptent   -> plus proche voisin : B (jeune)


---

# 📎 ANNEXE B — Preuve de la formule du risque de Bayes

---

**Objectif.** Montrer rigoureusement que
$$L^\star = L(g^\star) = \mathbb{E}\big[\min\big(\eta(X),\, 1 - \eta(X)\big)\big],$$
et que $g^\star(x) = \mathbb{1}\{\eta(x) \ge \tfrac{1}{2}\}$ est bien **optimal** (aucun classifieur ne fait mieux).

### Étape 1 — Conditionnement sur $X$

Pour un classifieur **quelconque** $g$, le risque 0–1 s'écrit, par la formule de l'espérance totale :
$$L(g) = \mathbb{P}\big(g(X) \neq Y\big) = \mathbb{E}\Big[\, \mathbb{1}\{g(X) \neq Y\} \,\Big]
= \mathbb{E}\Big[\, \underbrace{\mathbb{P}\big(g(X) \neq Y \mid X\big)}_{\text{erreur locale en } X} \,\Big].$$
Il suffit donc de **minimiser l'erreur en chaque point** $x$, puis de moyenner sur la loi de $X$.

### Étape 2 — Erreur locale en un point $x$

En un point $x$ fixé, la prédiction $g(x) \in \{0,1\}$ est une constante. Comme
$\eta(x) = \mathbb{P}(Y = 1 \mid X = x)$ :

$$\mathbb{P}\big(g(X) \neq Y \mid X = x\big) =
\begin{cases}
\mathbb{P}(Y = 0 \mid X = x) = 1 - \eta(x) & \text{si } g(x) = 1,\\[4pt]
\mathbb{P}(Y = 1 \mid X = x) = \eta(x) & \text{si } g(x) = 0.
\end{cases}$$

De façon compacte :
$$\mathbb{P}\big(g(X) \neq Y \mid X = x\big) = \mathbb{1}\{g(x)=1\}\,(1-\eta(x)) + \mathbb{1}\{g(x)=0\}\,\eta(x).$$

### Étape 3 — Le meilleur choix ponctuel est $g^\star$

En $x$, on choisit entre deux valeurs d'erreur possibles : $\eta(x)$ (si on prédit $0$) ou $1-\eta(x)$ (si on prédit $1$).
Le minimum des deux vaut par définition $\min(\eta(x),\,1-\eta(x))$, et il est atteint en prédisant la **classe la plus probable** :
$$g^\star(x) = \mathbb{1}\{\eta(x) \ge \tfrac{1}{2}\}
\quad\Longrightarrow\quad
\mathbb{P}\big(g^\star(X) \neq Y \mid X = x\big) = \min\big(\eta(x),\,1-\eta(x)\big).$$

*Vérification :* si $\eta(x) \ge 1/2$ alors $1-\eta(x) \le \eta(x)$, on prédit $1$ et l'erreur est $1-\eta(x) = \min$. Si $\eta(x) < 1/2$, on prédit $0$ et l'erreur est $\eta(x) = \min$. ✓

### Étape 4 — Optimalité et conclusion

Pour **tout** classifieur $g$ et tout $x$, on a par définition du minimum :
$$\mathbb{P}\big(g(X) \neq Y \mid X = x\big) \;\ge\; \min\big(\eta(x),\,1-\eta(x)\big) = \mathbb{P}\big(g^\star(X) \neq Y \mid X = x\big).$$
En prenant l'espérance sur $X$ (Étape 1), l'inégalité est **préservée** :
$$L(g) \;\ge\; \mathbb{E}\big[\min(\eta(X),\,1-\eta(X))\big] = L(g^\star) = L^\star.$$

Ainsi $g^\star$ minimise le risque, et son risque vaut exactement
$$\boxed{\,L^\star = \mathbb{E}\big[\min\big(\eta(X),\,1 - \eta(X)\big)\big].\,}$$

> **Remarque utile.** On peut aussi écrire $\min(\eta, 1-\eta) = \tfrac{1}{2} - \big|\eta - \tfrac{1}{2}\big|$, d'où
> $L^\star = \tfrac{1}{2} - \mathbb{E}\big|\eta(X) - \tfrac{1}{2}\big|$ : le risque est **minimal** quand $\eta$ est loin de $1/2$
> (classes séparées) et **maximal** ($= 1/2$) quand $\eta \equiv 1/2$ (bruit pur).


---

# 📎 ANNEXE C — Si le classifieur de Bayes minimise le risque, pourquoi n'est-il pas « le modèle le plus performant » ?

---

C'est une question apparemment paradoxale : $g^\star$ **est** le classifieur optimal (Annexe B), et pourtant on
ne peut pas dire que c'est « le meilleur modèle » au sens usuel. Trois raisons, à bien distinguer.

### 1. « Optimal » ne veut pas dire « parfait » — l'erreur $L^\star$ est incompressible

$g^\star$ atteint le **plus petit risque possible**, mais ce risque n'est en général **pas nul** :
$$L^\star = \mathbb{E}\big[\min(\eta(X), 1-\eta(X))\big] > 0 \quad \text{dès que } \eta(x) \notin \{0, 1\}.$$
Cette erreur vient du **bruit intrinsèque** du problème (les classes se recouvrent : au même $x$ on peut observer
$Y=0$ ou $Y=1$). C'est de l'**incertitude aléatoire** (aléa irréductible), pas un défaut du modèle. Exemple : au
Cas 2, même le classifieur parfait se trompe **30 %** du temps. Aucun modèle — aussi sophistiqué soit-il — ne peut
descendre sous $L^\star$. Donc « le plus performant » n'a jamais signifié « sans erreur ».

### 2. $g^\star$ n'est **pas un modèle** que l'on peut entraîner — c'est une référence théorique

$g^\star$ dépend de la vraie fonction $\eta(x) = \mathbb{P}(Y=1 \mid X=x)$, c'est-à-dire de la **loi jointe
$(X,Y)$**, qui est **inconnue** en pratique. On ne dispose que d'un **échantillon fini**. On construit donc un
estimateur $\hat{g}_n$ (k-NN, régression logistique, arbres…) qui **approche** $g^\star$. Son risque se décompose :
$$\underbrace{L(\hat{g}_n) - L^\star}_{\text{excès de risque} \,\ge\, 0} \;=\;
\underbrace{\text{biais d'approximation}}_{\text{la classe de modèles contient-elle } g^\star ?} \;+\;
\underbrace{\text{erreur d'estimation}}_{\text{données finies, variance}}.$$
$g^\star$ est la **cible idéale** (le « poteau d'arrivée ») servant à mesurer la qualité d'un modèle réel, mais on
ne peut **pas l'utiliser** directement : il n'est pas calculable à partir des données. C'est un **benchmark**, pas
un concurrent.

### 3. « Performant » dépend du **critère** — $g^\star$ optimise le coût 0–1, pas les autres

Le classifieur de Bayes avec seuil $1/2$ minimise le **taux d'erreur** (coût 0–1). Mais « performant » peut vouloir
dire autre chose selon le problème :
- **coûts asymétriques** (un faux négatif médical coûte plus cher qu'un faux positif) ⇒ le seuil optimal n'est
  plus $1/2$ mais dépend du rapport des coûts ;
- **classes déséquilibrées** ⇒ on regarde plutôt le rappel, la précision, le F1, la balanced accuracy… ;
- **problème de score/classement** (AUC-ROC) ⇒ on évalue l'ordonnancement, pas une décision binaire.

Pour ces critères, la règle « prédire la classe la plus probable » n'est **pas** celle qui optimise la métrique
visée. $g^\star$ est optimal **uniquement pour le risque 0–1** qui le définit.

### En résumé

| Affirmation | Vrai / Faux | Pourquoi |
|---|---|---|
| $g^\star$ minimise le risque 0–1 | ✅ Vrai | Démontré en Annexe B |
| $g^\star$ ne se trompe jamais | ❌ Faux | Erreur incompressible $L^\star > 0$ (bruit) |
| $g^\star$ est un modèle qu'on peut entraîner | ❌ Faux | Il dépend de $\eta$ inconnue ; c'est une référence théorique |
| $g^\star$ est le meilleur pour toute métrique | ❌ Faux | Optimal pour le coût 0–1 seulement (pas coûts asymétriques, F1, AUC…) |

**Conclusion.** Le classifieur de Bayes est le **meilleur possible pour le taux d'erreur**, ce qui en fait la borne
de référence — mais il reste **imparfait** (à cause du bruit), **inaccessible** (loi inconnue) et **spécifique à une
métrique**. « Meilleur classifieur théorique » ≠ « meilleur modèle utilisable en pratique ».
